# Astro II Lab 4 — Inflation & The Horizon Problem

**Goal:** Understand the Horizon Problem in the standard Big Bang cosmology and explore how an early period of exponential expansion (Inflation) elegantly solves it.

---

## Part 1: The Comoving Hubble Radius

The **Comoving Hubble Radius**, given by $r_H = a
rac{c}{aH}$, is a crucial concept. It represents the maximum comoving distance that particles can communicate over at a given epoch. If $r_H$ is shrinking, the observable universe is getting smaller in comoving coordinates (modes are "exiting" the horizon). If $r_H$ is growing, the observable universe is getting larger (modes are "entering" the horizon).

In the Standard Big Bang model (assuming radiation and then matter domination):
1. **Radiation Dominated ($w = 1/3$):** $H \propto a^{-2} \implies r_H \propto a$. The comoving Hubble radius grows.
2. **Matter Dominated ($w = 0$):** $H \propto a^{-3/2} \implies r_H \propto a^{1/2}$. The comoving Hubble radius also grows.

Because $r_H$ always grew in the early universe, regions we see today in the Cosmic Microwave Background (CMB) at opposite sides of the sky were never in causal contact. Yet, they have exactly the same temperature! This is the **Horizon Problem**.

**Inflation** posits a period of accelerated expansion ($w \approx -1$) very early on. 
During inflation:
- **Inflation ($w \approx -1$):** $H \approx 	ext{const} \implies r_H \propto a^{-1}$. The comoving Hubble radius shrinks drastically!

**What to do:**
1. Run the cell below to load the interactive tool.
2. Adjust the number of $e$-folds of inflation ($N$) and the energy scale to see how the comoving Hubble radius behaves.
3. Observe how inflation forces modes to "exit" the horizon early on, allowing the entire observable universe to have been in causal contact before inflation began.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Constants
a_0 = 1.0
H_0 = 70.0 # km/s/Mpc
c = 2.9979e5 # km/s
r_H0_Mpc = c / H_0 # ~ 4280 Mpc

# For simplicity, we define early universe epochs by temperature/energy scale
# Reheating temperature (end of inflation) T_reheat
# Matter-radiation equality a_eq ~ 1/3400 (z_eq ~ 3399)
a_eq = 1.0 / 3400.0

def plot_horizon(N_efolds, log_T_reheat):
    T_reheat = 10.0**float(log_T_reheat) # GeV
    # Relationship between temperature and scale factor during radiation domination
    # T_0 ~ 2.73 K ~ 2.3e-13 GeV
    T_0 = 2.35e-13 # GeV
    
    # Scale factor at end of inflation (assuming immediate reheating)
    a_end_inf = T_0 / T_reheat
    log_a_end_inf = np.log10(a_end_inf)
    
    # Scale factor array (log scale) to cover the whole range
    log_a = np.linspace(log_a_end_inf - N_efolds - 10, 0, 1000)
    a = 10**log_a
    
    # We want to plot the comoving Hubble radius: r_H = c / (a * H)
    # We normalize to today: at a=1, r_H = r_H0_Mpc
    # During Matter Domination (a_eq < a < 1): H ~ a^(-3/2) -> aH ~ a^(-1/2) -> r_H \propto a^(1/2)
    # During Radiation Domination (a_end_inf < a < a_eq): H ~ a^(-2) -> aH ~ a^(-1) -> r_H \propto a
    # During Inflation (a < a_end_inf): H ~ constant -> aH ~ a -> r_H \propto a^(-1)
    
    c_aH = np.zeros_like(a)
    
    for i, a_val in enumerate(a):
        if a_val > a_eq:
            # Matter domination
            # r_H(a) = r_H0 * a^(1/2)
            c_aH[i] = r_H0_Mpc * np.sqrt(a_val)
        elif a_val > a_end_inf:
            # Radiation domination
            # Match boundary at a_eq: r_H(a_eq) = r_H0 * a_eq^(1/2)
            # r_H(a) = r_H(a_eq) * (a / a_eq)
            r_H_eq = r_H0_Mpc * np.sqrt(a_eq)
            c_aH[i] = r_H_eq * (a_val / a_eq)
        else:
            # Inflation
            # Match boundary at a_end_inf
            r_H_eq = r_H0_Mpc * np.sqrt(a_eq)
            r_H_end_inf = r_H_eq * (a_end_inf / a_eq)
            # r_H(a) = r_H(a_end_inf) * (a_end_inf / a)
            c_aH[i] = r_H_end_inf * (a_end_inf / a_val)
            
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(log_a, np.log10(c_aH), lw=3, color='#E91E63', label='Comoving Hubble Radius $\log_{10}(c/aH)$')
    
    # Add scales
    observable_universe = np.log10(r_H0_Mpc)
    ax.axhline(observable_universe, color='gray', linestyle='--', label='Observable Universe Today (~4.3 Gpc)')
    
    # The scale that entered the horizon today corresponds to r_H0. 
    # Let's see at what scale factor it exited the horizon during inflation.
    
    ax.axvspan(log_a_end_inf - N_efolds, log_a_end_inf, color='orange', alpha=0.2, label='Inflation Period')
    ax.axvspan(log_a_end_inf, np.log10(a_eq), color='cyan', alpha=0.1, label='Radiation Domination')
    ax.axvspan(np.log10(a_eq), 0, color='blue', alpha=0.1, label='Matter/$\Lambda$ Domination')
    
    ax.set_xlabel('Scale Factor $\log_{10}(a)$', fontsize=12)
    ax.set_ylabel('Comoving Radius $\log_{10}(r_H / \mathrm{Mpc})$', fontsize=12)
    ax.set_xlim(log_a_end_inf - N_efolds - 5, 0)
    
    ax.set_ylim(-30, observable_universe + 5)
    
    ax.legend(fontsize=11, loc='lower left')
    ax.grid(True, alpha=0.3, ls=':')
    T_reheat = 10.0**float(log_T_reheat) # GeV
    
    # Calculate required N_efolds
    r_H_eq = r_H0_Mpc * np.sqrt(a_eq)
    r_H_end_inf = r_H_eq * (a_end_inf / a_eq)
    
    # We need r_H_start_inf < r_H0_Mpc
    # r_H_start_inf = r_H_end_inf * (a_end_inf / a_start_inf) = r_H_end_inf * exp(N_efolds)
    # exp(N) > r_H0 / r_H_end_inf 
    # N > ln(r_H0 / r_H_end_inf)
    N_required = np.log(r_H0_Mpc / r_H_end_inf)
    
    if N_efolds < N_required:
       plt.text(log_a_end_inf - N_efolds + 2, observable_universe - 3, 
                f"Horizon Problem Not Solved!\nNeed N > {N_required:.1f}", 
                color='red', fontsize=12, weight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='red'))
    else:
       plt.text(log_a_end_inf - N_efolds + 2, observable_universe - 3, 
                f"Horizon Problem Solved!\nN > {N_required:.1f} achieved", 
                color='green', fontsize=12, weight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='green'))
    
    plt.tight_layout()
    plt.show()

w_N = widgets.IntSlider(value=60, min=20, max=100, step=2, description='N (e-folds)')
w_Treheat = widgets.IntSlider(value=15, min=5, max=16, step=1, description='log(T_reheat) GeV')

ui = widgets.VBox([
    widgets.HTML('<b>Inflation Parameters</b>'),
    widgets.HBox([w_N, w_Treheat])
])

out = widgets.interactive_output(plot_horizon, {'N_efolds': w_N, 'log_T_reheat': w_Treheat})
display(ui, out)


Output()

### Discussion points (Horizon Problem)
1. **The Origin of the Horizon Problem:** Before inflation was introduced, what was the primary theoretical gap explaining the uniformity of the CMB? 
2. **Minimum e-folds:** Based on the interactive plot (and basic cosmological principles), roughly how many $e$-folds of inflation ($N$) are required at a minimum to solve the horizon problem for our observable universe? What happens to the largest scales if $N$ is too small?
3. **Shrinking Hubble Sphere:** Explain in your own words what $r_H = c/(aH)$ decreasing means for a patch of space. How does a region that was originally causally connected become disconnected during inflation?


---

## Part 2: Causality and the Physical Hubble Radius

While the comoving Hubble radius ($r_H$) is crucial for the mathematics, it's often more intuitive to look at **physical (proper) scales**. 

The **Physical Hubble Radius** is simply $R_H(t) = c/H(t)$. It sets the physical distance light can travel in a Hubble time. 
A physical wavelength, or scale of a perturbation, $\lambda_{\mathrm{phys}}$, simply stretches with the expansion of the universe: $\lambda_{\mathrm{phys}}(t) \propto a(t)$.

1. **Sub-Horizon (Causally Connected):** When $\lambda_{\mathrm{phys}} < c/H$, the scale is smaller than the horizon. Microphysical processes can act on this scale.
2. **Super-Horizon (Causally Disconnected):** When $\lambda_{\mathrm{phys}} > c/H$, the scale is larger than the horizon. The perturbation is effectively "frozen in" because information cannot travel across it.

In standard cosmology (radiation/matter domination):
- $R_H = c/H$ grows faster than $a(t)$.
- Thus, the horizon expands *faster* than physical scales stretch. Scales that were initially super-horizon will eventually **"re-enter"** the horizon.

During Inflation:
- $H \approx$ const, so $R_H \approx$ const. The physical horizon is stationary.
- Physical scales still stretch exponentially: $\lambda_{\mathrm{phys}} \propto a(t) \propto e^{Ht}$. 
- Thus, microscopic scales start sub-horizon, but get stretched so violently they **"exit"** the horizon and become frozen.

**What to do:**
1. Use the slider to select a comoving scale $k$ (which determines the size of the physical wavelength).
2. Watch how the scale starts inside the horizon during inflation, is forced outside the horizon, and then naturally re-enters during standard cosmic expansion.



In [5]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Constants (same as before)
a_0 = 1.0
H_0 = 70.0 # km/s/Mpc
c = 2.9979e5 # km/s
R_H0_Mpc = c / H_0 # ~ 4280 Mpc

# Epoch boundary
a_eq = 1.0 / 3400.0

def plot_physical_scales(log_T_reheat, log_k_scale):
    # This function uses log_T_reheat directly (since it works in previous cell now)
    T_reheat = 10.0**float(log_T_reheat)
    T_0 = 2.35e-13 # GeV
    
    a_end_inf = T_0 / T_reheat
    log_a_end_inf = np.log10(a_end_inf)
    
    N_efolds = 65 # Fix N to a generous amount so it's always solved here
    
    log_a = np.linspace(log_a_end_inf - N_efolds, 0, 1000)
    a = 10**log_a
    
    # 1. Calculate Physical Hubble Radius: R_H = c/H
    # Remember H(a) scaling:
    # Matter: H \propto a^(-3/2) -> R_H \propto a^(3/2)
    # Radiation: H \propto a^(-2) -> R_H \propto a^2
    # Inflation: H pprox const -> R_H = constant
    
    R_H = np.zeros_like(a)
    
    for i, a_val in enumerate(a):
        if a_val > a_eq:
            # R_H0 * a^(3/2)
            R_H[i] = R_H0_Mpc * a_val**(1.5)
        elif a_val > a_end_inf:
            # R_H(a_eq) * (a / a_eq)^2
            R_H_eq = R_H0_Mpc * a_eq**(1.5)
            R_H[i] = R_H_eq * (a_val / a_eq)**2
        else:
            # Constant during inflation
            R_H_eq = R_H0_Mpc * a_eq**(1.5)
            R_H_end_inf = R_H_eq * (a_end_inf / a_eq)**2
            R_H[i] = R_H_end_inf
            
            
    # 2. Calculate Physical Scale: lambda_phys \propto a
    # The comoving scale lambda_comoving is constant.
    # lambda_phys = lambda_comoving * a
    # We use a dimensionless slider 'log_k_scale' to pick different lambda_comoving
    lambda_comoving = 10.0**float(log_k_scale)
    lambda_phys = lambda_comoving * a
    
    # Plotting
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(log_a, np.log10(R_H), lw=3, color='black', label='Physical Hubble Radius $\log_{10}(R_H / \mathrm{Mpc})$')
    ax.plot(log_a, np.log10(lambda_phys), lw=3, color='orange', ls='--', label=f'Physical Wavelength $\log_{10}(\lambda_{{phys}})$')
    
    # Shade causal regions
    # When lambda < R_H, we are Sub-Horizon
    ax.fill_between(log_a, -40, np.log10(R_H), color='green', alpha=0.1, label='Sub-Horizon (Causally Connected)')
    # When lambda > R_H, we are Super-Horizon
    ax.fill_between(log_a, np.log10(R_H), 15, color='red', alpha=0.1, label='Super-Horizon (Frozen Out)')
    
    
    ax.axvspan(log_a_end_inf - N_efolds, log_a_end_inf, color='orange', alpha=0.2, label='Inflation Period')
    ax.axvspan(log_a_end_inf, np.log10(a_eq), color='cyan', alpha=0.1, label='Radiation Domination')
    ax.axvspan(np.log10(a_eq), 0, color='blue', alpha=0.1, label='Matter/$\Lambda$ Domination')
    
    ax.set_xlabel('Scale Factor $\log_{10}(a)$', fontsize=12)
    ax.set_ylabel('Physical Scale $\log_{10}(\mathrm{Mpc})$', fontsize=12)
    ax.set_xlim(log_a_end_inf - N_efolds, 0)
    ax.set_ylim(-40, 10)
    
    # To keep the legend manageable, we'll put it outside
    ax.legend(fontsize=10, loc='center left', bbox_to_anchor=(1, 0.5))
    ax.grid(True, alpha=0.3, ls=':')
    ax.set_title('Physical Scales: Exiting & Re-Entering the Horizon', fontsize=14, weight='bold')
    
    plt.tight_layout()
    plt.show()

w_k_scale = widgets.FloatSlider(value=3.5, min=-10, max=10, step=0.5, description='log(k scale)')
w_T_reheat_2 = widgets.IntSlider(value=15, min=5, max=16, step=1, description='log(T_reheat)')

ui2 = widgets.VBox([
    widgets.HTML('<b>Adjust Perturbation Scale</b>'),
    widgets.HBox([w_k_scale, w_T_reheat_2])
])

out2 = widgets.interactive_output(plot_physical_scales, {'log_T_reheat': w_T_reheat_2, 'log_k_scale': w_k_scale})
display(ui2, out2)



Output()